# Análisis de Errores — Modelos Predictivos TDAH

| | |
|---|---|
| **Dataset** | `data/processed/processed_data.csv` |
| **Registros** | 875 · Split 80/20 estratificado (random_state=42) |
| **Modelos** | Random Forest (300 árboles, class_weight='balanced') |
| **Objetivo** | Identificar patrones de error, casos ambiguos y limitaciones clínicas |

**¿Por qué analizar los errores?** Un modelo con F1-macro > 0.95 comete pocos errores, pero esos errores son diagnósticamente informativos. Revelan qué perfiles clínicos están en la frontera entre subtipos, dónde las features disponibles no son suficientes, y qué casos requieren mayor atención clínica.

### Tabla de contenidos
1. Configuración y entrenamiento de modelos
2. Análisis de errores — Modelo 4 clases
3. Análisis de errores — Modelo 13 clases
4. Síntesis comparativa y conclusiones

---
> **Reproducibilidad:** este notebook usa `random_state=42` en el split y en los modelos RF, idéntico al usado en `model_4clases.ipynb` y `model_13clases.ipynb`. Los resultados son exactamente los mismos conjuntos train/test.

---
## Bloque 1 — Configuración, datos y entrenamiento

Se reconstruyen los mismos modelos RF de los notebooks de modelado usando el split estratificado idéntico (`random_state=42`). Esto garantiza que el análisis de errores corresponde exactamente a las evaluaciones reportadas.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_recall_fscore_support)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi'        : 150,
    'savefig.dpi'       : 150,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.titleweight'  : 'bold',
    'axes.titlesize'    : 13,
    'axes.labelsize'    : 11,
    'xtick.labelsize'   : 10,
    'ytick.labelsize'   : 10,
    'legend.fontsize'   : 9,
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : 'white',
    'grid.alpha'        : 0.35,
})

C_BLUE = '#4C72B0'; C_ORG = '#DD8452'; C_GRN = '#55A868'
C_RED  = '#C44E52'; C_GRAY = '#8C8C8C'; C_PURP = '#8172B2'
C_OK   = '#2ca02c'; C_ERR  = '#d62728'

FEATURE_LABELS = {
    'problemas_atencion_basc3' : 'Problemas de atención (BASC-3)',
    'hiperactividad_basc3'     : 'Hiperactividad (BASC-3)',
    'problemas_conducta_basc3' : 'Problemas de conducta (BASC-3)',
    'agresividad_basc3'        : 'Agresividad (BASC-3)',
    'atipicidad_basc3'         : 'Atipicidad (BASC-3)',
    'cit'                      : 'Coef. Intelectual (CIT)',
    'fluidez_fonologica'       : 'Fluidez fonológica',
    'antecedentes_familiares'  : 'Antecedentes familiares',
    'retrasos_desarrollo'      : 'Retrasos del desarrollo',
    'antecedentes_obstetricos' : 'Antecedentes obstétricos',
    'edad'                     : 'Edad',
    'escolaridad'              : 'Escolaridad',
    'sexo'                     : 'Sexo',
    'lateralidad'              : 'Lateralidad',
}
CONT_FEATS = ['problemas_atencion_basc3', 'hiperactividad_basc3',
              'problemas_conducta_basc3', 'agresividad_basc3', 'atipicidad_basc3',
              'cit', 'fluidez_fonologica']
print("Setup completado.")


In [ ]:
df = pd.read_csv('../data/processed/processed_data.csv', encoding='utf-8-sig')
print(f"Dataset cargado: {df.shape[0]} registros × {df.shape[1]} columnas")

FEATURES = ['edad', 'sexo', 'escolaridad', 'lateralidad',
            'antecedentes_familiares', 'antecedentes_obstetricos', 'retrasos_desarrollo',
            'cit', 'agresividad_basc3', 'hiperactividad_basc3',
            'problemas_conducta_basc3', 'problemas_atencion_basc3',
            'atipicidad_basc3', 'fluidez_fonologica']
FEATURES_CAT = ['sexo', 'lateralidad', 'antecedentes_familiares',
                'antecedentes_obstetricos', 'retrasos_desarrollo']

# ── Codificación 4 clases ─────────────────────────────────────────────────
def tipo4(e):
    e = str(e).lower()
    if 'típico'      in e: return 'Desarrollo típico'
    if 'inatento'    in e: return 'TDAH inatento'
    if 'hiperactivo' in e: return 'TDAH hiperactivo/impulsivo'
    return 'TDAH combinado'

df['tipo_tdah'] = df['etiqueta'].apply(tipo4)

# ── Encoding ──────────────────────────────────────────────────────────────
X = df[FEATURES].copy()
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[FEATURES_CAT] = enc.fit_transform(X[FEATURES_CAT])

le4  = LabelEncoder()
y4   = le4.fit_transform(df['tipo_tdah'])
le13 = LabelEncoder()
y13  = le13.fit_transform(df['etiqueta'])

print(f"\n4 clases:  {dict(zip(le4.classes_,  np.bincount(y4)))}")
print(f"\n13 clases: {dict(zip(le13.classes_, np.bincount(y13)))}")


In [ ]:
# ── Split estratificado idéntico al de los notebooks de modelado ──────────
# random_state=42 garantiza exactamente los mismos conjuntos de train/test
X_tr4,  X_te4,  y_tr4,  y_te4  = train_test_split(X, y4,  test_size=0.2,
                                                    stratify=y4,  random_state=42)
X_tr13, X_te13, y_tr13, y_te13 = train_test_split(X, y13, test_size=0.2,
                                                    stratify=y13, random_state=42)

# ── Entrenar Random Forest (mismo config que model notebooks) ────────────
rf4  = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                               random_state=42, n_jobs=-1)
rf13 = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                               random_state=42, n_jobs=-1)
rf4.fit(X_tr4,   y_tr4)
rf13.fit(X_tr13, y_tr13)

# ── Predicciones y probabilidades ────────────────────────────────────────
pred4   = rf4.predict(X_te4)
proba4  = rf4.predict_proba(X_te4)
pred13  = rf13.predict(X_te13)
proba13 = rf13.predict_proba(X_te13)

f1_4  = f1_score(y_te4,  pred4,  average='macro')
f1_13 = f1_score(y_te13, pred13, average='macro')

print(f"RF 4 clases  — F1-macro test: {f1_4:.4f}")
print(f"RF 13 clases — F1-macro test: {f1_13:.4f}")
print(f"\nCasos test 4 clases:  {len(y_te4)}")
print(f"Casos test 13 clases: {len(y_te13)}")


In [ ]:
# ── Helper: heatmap de métricas por clase ────────────────────────────────
def plot_class_metrics(y_true, y_pred, labels, title):
    p, r, f, sup = precision_recall_fscore_support(y_true, y_pred, labels=np.arange(len(labels)))
    met = pd.DataFrame({'Precisión': p, 'Recall': r, 'F1': f, 'Soporte': sup.astype(int)},
                        index=labels)
    n = len(labels)
    fig, axes = plt.subplots(1, 2, figsize=(14, max(3.5, n * 0.45 + 1.5)),
                              gridspec_kw={'width_ratios': [3, 1]})

    # Heatmap P/R/F1
    ax = axes[0]
    heat = met[['Precisión', 'Recall', 'F1']].astype(float)
    im = ax.imshow(heat.values, cmap='RdYlGn', vmin=0.4, vmax=1.0, aspect='auto')
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(['Precisión', 'Recall', 'F1'], fontsize=11)
    ax.set_yticks(range(n)); ax.set_yticklabels(labels, fontsize=9.5)
    for i in range(n):
        for j, col in enumerate(['Precisión', 'Recall', 'F1']):
            v = heat.iloc[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    fontsize=9.5, color='black' if 0.4 < v < 0.85 else 'white', fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    ax.set_title(f'{title}\nPrecisión · Recall · F1 por clase', pad=8)

    # Barras de soporte
    ax2 = axes[1]
    colors = plt.cm.Blues(np.linspace(0.35, 0.75, n))
    bars = ax2.barh(range(n), met['Soporte'], color=colors, edgecolor='white', height=0.65)
    for bar, val in zip(bars, met['Soporte']):
        ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=9)
    ax2.set_yticks(range(n)); ax2.set_yticklabels([])
    ax2.set_xlabel('Casos en test', fontsize=10)
    ax2.set_title('Soporte', pad=8)
    plt.tight_layout()
    plt.show()
    return met

# ── Helper: pares de confusión ────────────────────────────────────────────
def plot_confusion_pairs(y_true, y_pred, labels, title, top_n=10):
    mask = y_true != y_pred
    err_true  = [labels[t] for t in y_true[mask]]
    err_pred  = [labels[p] for p in y_pred[mask]]
    pairs = pd.Series([f'{t}  →  {p}' for t, p in zip(err_true, err_pred)])
    pairs = pairs.value_counts().head(top_n)

    fig, ax = plt.subplots(figsize=(11, max(3.5, len(pairs) * 0.55 + 1)))
    colors = plt.cm.OrRd(np.linspace(0.35, 0.80, len(pairs)))[::-1]
    bars = ax.barh(range(len(pairs)), pairs.values, color=colors, edgecolor='white', height=0.65)
    for bar, val in zip(bars, pairs.values):
        ax.text(val + 0.1, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=10, fontweight='bold')
    ax.set_yticks(range(len(pairs)))
    ax.set_yticklabels(pairs.index, fontsize=9.5)
    ax.set_xlabel('Número de casos', fontsize=10)
    ax.set_title(f'{title}\nPares de confusión más frecuentes  '
                 f'(Real  →  Predicha) — Top {len(pairs)} de {mask.sum()} errores', pad=8)
    plt.tight_layout()
    plt.show()
    total = mask.sum()
    top_pct = pairs.values[0] / total * 100 if len(pairs) > 0 else 0
    print(f"Total errores: {total} / {len(y_true)} ({total/len(y_true)*100:.1f}%)")
    print(f"Par más frecuente: '{pairs.index[0]}' — {pairs.values[0]} casos ({top_pct:.1f}% de los errores)")

# ── Helper: confianza correctos vs errores ────────────────────────────────
def plot_confidence(y_true, y_pred, proba, labels, title):
    max_p   = proba.max(axis=1)
    correct = y_true == y_pred

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Panel 1: distribución KDE
    ax = axes[0]
    for ok, label, color in [(True, 'Correcto', C_OK), (False, 'Error', C_ERR)]:
        vals = max_p[correct == ok]
        ax.hist(vals, bins=20, alpha=0.45, color=color, density=True, label=f'{label} (n={ok.sum() if hasattr(ok,"sum") else (correct==ok).sum()})')
        from scipy.stats import gaussian_kde
        if len(vals) > 2:
            kde = gaussian_kde(vals, bw_method=0.3)
            xs  = np.linspace(0, 1, 300)
            ax.plot(xs, kde(xs), color=color, lw=2)
    ax.axvline(max_p[correct].mean(),  color=C_OK,  lw=1.5, linestyle='--',
               label=f'Media correctos = {max_p[correct].mean():.3f}')
    ax.axvline(max_p[~correct].mean(), color=C_ERR, lw=1.5, linestyle='--',
               label=f'Media errores = {max_p[~correct].mean():.3f}')
    ax.set_xlabel('Probabilidad máxima asignada por el modelo', fontsize=10)
    ax.set_ylabel('Densidad', fontsize=10)
    ax.set_title(f'{title}\nDistribución de confianza', pad=8)
    ax.legend(fontsize=8.5)

    # Panel 2: confianza media por clase (correctos vs errores)
    ax2 = axes[1]
    n_cls = len(labels)
    cl_ok  = [max_p[(correct) & (y_true == i)].mean()  if (correct & (y_true==i)).sum() > 0 else np.nan for i in range(n_cls)]
    cl_err = [max_p[(~correct) & (y_true == i)].mean() if (~correct & (y_true==i)).sum() > 0 else np.nan for i in range(n_cls)]
    x = np.arange(n_cls)
    w = 0.35
    ax2.bar(x - w/2, cl_ok,  width=w, color=C_OK,  alpha=0.8, label='Correctos', edgecolor='white')
    ax2.bar(x + w/2, cl_err, width=w, color=C_ERR, alpha=0.8, label='Errores',   edgecolor='white')
    ax2.set_xticks(x)
    ax2.set_xticklabels(labels, rotation=25, ha='right', fontsize=9)
    ax2.set_ylim(0, 1.1)
    ax2.set_ylabel('Confianza media', fontsize=10)
    ax2.set_title('Confianza media por clase', pad=8)
    ax2.legend(fontsize=9)
    for i, (ok_v, err_v) in enumerate(zip(cl_ok, cl_err)):
        if not np.isnan(ok_v):
            ax2.text(i - w/2, ok_v + 0.02, f'{ok_v:.2f}', ha='center', fontsize=8)
        if not np.isnan(err_v):
            ax2.text(i + w/2, err_v + 0.02, f'{err_v:.2f}', ha='center', fontsize=8, color=C_ERR)
    plt.tight_layout()
    plt.show()

    # Casos de baja confianza
    low_conf = (max_p < 0.5) & (~correct)
    print(f"\nCasos mal clasificados con confianza < 0.50: {low_conf.sum()} "
          f"({low_conf.sum()/len(y_true)*100:.1f}% del total test)")
    high_conf_err = (max_p > 0.80) & (~correct)
    print(f"Casos mal clasificados con alta confianza (>0.80): {high_conf_err.sum()} "
          f"— errores difíciles de detectar")

# ── Helper: perfil clínico de mal clasificados ────────────────────────────
def plot_error_profiles(y_true, y_pred, X_test_raw, labels, title):
    \"\"\"
    Heatmap z-score: compara perfil medio de correctos vs mal clasificados
    para cada clase real. Solo usa variables continuas.
    \"\"\"
    cont = [f for f in CONT_FEATS if f in X_test_raw.columns]
    cont_labels = [FEATURE_LABELS.get(f, f) for f in cont]

    # Z-score sobre el test set completo
    mu  = X_test_raw[cont].mean()
    sig = X_test_raw[cont].std().replace(0, 1)
    Xz  = (X_test_raw[cont] - mu) / sig

    # Construir tabla: fila = (clase, grupo), cols = variables
    rows, row_labels, row_colors = [], [], []
    for i, lbl in enumerate(labels):
        mask_cls = (y_true == i)
        ok_mask  = mask_cls & (y_true == y_pred)
        err_mask = mask_cls & (y_true != y_pred)
        n_ok  = ok_mask.sum()
        n_err = err_mask.sum()
        if n_ok > 0:
            rows.append(Xz[ok_mask].mean().values)
            row_labels.append(f'{lbl}\n✔ correctos (n={n_ok})')
            row_colors.append(C_OK)
        if n_err > 0:
            rows.append(Xz[err_mask].mean().values)
            row_labels.append(f'{lbl}\n✖ errores (n={n_err})')
            row_colors.append(C_ERR)

    if not rows:
        print("No hay errores suficientes para el perfil.")
        return

    heat = pd.DataFrame(rows, columns=cont_labels, index=range(len(rows)))
    n_rows = len(rows)
    fig, ax = plt.subplots(figsize=(max(10, len(cont) * 1.1), max(5, n_rows * 0.6 + 1.5)))
    im = ax.imshow(heat.values, cmap='RdBu_r', vmin=-2.5, vmax=2.5, aspect='auto')
    ax.set_xticks(range(len(cont_labels)))
    ax.set_xticklabels(cont_labels, rotation=35, ha='right', fontsize=9)
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(row_labels, fontsize=8.5)
    for i in range(n_rows):
        for j in range(len(cont)):
            v = heat.iloc[i, j]
            ax.text(j, i, f'{v:+.2f}', ha='center', va='center',
                    fontsize=7.5, color='white' if abs(v) > 1.4 else 'black')
    # Línea separadora entre clases
    prev_lbl = None
    for ri, rl in enumerate(row_labels):
        cur_lbl = rl.split('\n')[0]
        if cur_lbl != prev_lbl and ri > 0:
            ax.axhline(ri - 0.5, color='black', lw=1.5)
        prev_lbl = cur_lbl
    # Marcadores de color en eje Y
    for ri, color in enumerate(row_colors):
        ax.add_patch(plt.Rectangle((-0.5, ri - 0.5), 0.3, 1,
                                   color=color, clip_on=False, transform=ax.transData))
    plt.colorbar(im, ax=ax, label='Z-score (desviaciones del promedio del test)', fraction=0.02, pad=0.01)
    ax.set_title(f'{title}\nPerfil clínico (z-score) — correctos vs mal clasificados por clase', pad=10)
    plt.tight_layout()
    plt.show()


📋 **Setup completo.** Ambos modelos RF entrenados con los mismos parámetros y split que en los notebooks de modelado. Los F1-macro impresos deben coincidir con los reportados en `model_4clases.ipynb` y `model_13clases.ipynb`.

---
## Bloque 2 — Modelo de 4 clases: análisis de errores

El modelo Random Forest alcanzó F1-macro = 0.97 en el test set (ver `model_4clases.ipynb`). Con ese nivel de rendimiento, los errores son escasos y altamente informativos: revelan los casos límite entre subtipos — los perfiles más ambiguos que el clínico también tiene mayor dificultad en clasificar.

Los cuatro subtipos son:
- **Desarrollo típico** — sin criterios diagnósticos TDAH
- **TDAH inatento** — predominio de síntomas de inatención
- **TDAH hiperactivo/impulsivo** — predominio de hiperactividad e impulsividad
- **TDAH combinado** — criterios completos en ambas dimensiones

---
### 2.1 Métricas por clase

Precisión, Recall y F1 por subtipo, con el soporte (nº de casos en test) como referencia para interpretar la fiabilidad de cada estimación.

In [ ]:
met4 = plot_class_metrics(y_te4, pred4, le4.classes_,
                          'RF — 4 clases')

📋 **Interpretación**

La tabla de métricas muestra el rendimiento del modelo desagregado por subtipo. En un modelo con F1-macro > 0.96, es esperable que todas las clases tengan Recall ≥ 0.90, pero las diferencias entre clases informan sobre qué subtipos son inherentemente más difíciles de separar.

El soporte por clase en test (~20% estratificado) refleja la distribución real de la muestra. Clases con menor soporte producen estimaciones menos estables — una diferencia de 2–3 casos puede cambiar el F1 en varios puntos porcentuales. Por eso, el análisis de errores focaliza en los *patrones* más que en los valores exactos.

---
### 2.2 Pares de confusión más frecuentes

¿Cuándo se equivoca el modelo, qué subtipo confunde con qué otro? Los pares Real → Predicha revelan la dirección del error y apuntan a qué dimensiones clínicas son más difíciles de separar.

In [ ]:
plot_confusion_pairs(y_te4, pred4, le4.classes_,
                     'RF — 4 clases')

📋 **Interpretación**

Los pares de confusión esperados en un modelo de TDAH son:
- **Inatento → Combinado** (o viceversa): casos con síntomas de inatención marcados y algo de hiperactividad que no llegan a criterio combinado.
- **Hiperactivo/impulsivo → Combinado**: casos límite en la dimensión de inatención.
- **Típico → cualquier TDAH**: casos subumbral con puntajes BASC-3 en zona de riesgo.

Un par *asimétrico* (ej. Inatento → Combinado más frecuente que Combinado → Inatento) indica que el modelo tiene un sesgo hacia la clase combinada, que es la más prevalente.

---
### 2.3 Confianza del modelo: correctos vs errores

La probabilidad máxima asignada por el modelo (`max_proba`) es una medida de confianza. Si los errores ocurren con alta confianza, el modelo está "seguro de equivocarse" — señal de que esos casos son genuinamente ambiguos o atípicos en la muestra.

In [ ]:
plot_confidence(y_te4, pred4, proba4, le4.classes_,
               'RF — 4 clases')

📋 **Interpretación**

En modelos de alta precisión, los errores suelen concentrarse en la zona de baja confianza (max_proba < 0.60), donde el modelo reparte probabilidad casi uniformemente entre dos o más clases. Estos son los casos genuinamente ambiguos.

Los errores con alta confianza (max_proba > 0.80) son más preocupantes: el modelo asigna erróneamente una probabilidad alta a la clase equivocada. En un contexto clínico, estos serían los casos más peligrosos porque no generan ninguna señal de incertidumbre al usuario.

---
### 2.4 Perfil clínico: correctos vs mal clasificados

El heatmap muestra el z-score promedio de las variables continuas para los casos correctamente clasificados (✔) y para los errores (✖), separado por clase real. Un z-score negativo (azul) indica valores por debajo de la media del test; positivo (rojo) indica por encima.

**Cómo leer:** si los errores de "TDAH inatento" tienen z-score de hiperactividad más alto que los correctos de la misma clase, eso indica que los casos mal clasificados son precisamente los que tienen síntomas mixtos — los que están en la frontera entre inatento y combinado.

In [ ]:
# Reconstruir DataFrame con nombres de columnas originales para el perfil
X_te4_df = pd.DataFrame(
    X_te4.values if hasattr(X_te4, 'values') else X_te4,
    columns=FEATURES
)
plot_error_profiles(y_te4, pred4, X_te4_df, le4.classes_,
                    'RF — 4 clases')

📋 **Interpretación**

Las diferencias entre perfiles ✔ y ✖ dentro de una misma clase revelan los rasgos que llevan al error:

- Si los errores de **TDAH inatento** tienen Hiperactividad (BASC-3) más alta que los correctos → el modelo confunde casos inatentos con componente hiperactivo elevado.
- Si los errores de **Desarrollo típico** tienen valores BASC-3 por encima del promedio → son perfiles subumbral (no diagnosticados, pero con sintomatología presente).
- Si los perfiles ✔ y ✖ son similares → el error no está en las variables disponibles sino en información no capturada (ej. contexto familiar, observación directa).

---
## Bloque 3 — Modelo de 13 clases: análisis de errores

El modelo RF de 13 clases alcanzó F1-macro = 0.95. Las 13 etiquetas codifican *subtipo TDAH × comorbilidad*: TND (Trastorno Negativista Desafiante), TEAZ (Espectro Autista subcrítico) y TEA (Trastorno del Espectro Autista). Distinguir entre, por ejemplo, *TDAH inatento* y *TDAH inatento + TND* es una tarea más fina que separar los 4 subtipos principales.

La hipótesis central de este análisis es que **los errores del modelo de 13 clases se producen principalmente dentro del mismo subtipo TDAH** — es decir, confunde la comorbilidad pero acierta el subtipo principal.

---
### 3.1 Métricas por clase

In [ ]:
met13 = plot_class_metrics(y_te13, pred13, le13.classes_,
                           'RF — 13 clases')

📋 **Interpretación**

Con 13 clases, el soporte por clase en test es reducido (entre 3 y ~25 casos), lo que hace las métricas individuales más ruidosas. Clases con soporte < 5 en test deben interpretarse con cautela — un solo caso mal clasificado mueve el F1 en más de 0.20 puntos.

El foco analítico debe centrarse en: (a) las clases con Recall más bajo — subtipos que el modelo "no detecta bien"; y (b) las clases con Precisión baja — clases donde el modelo produce falsos positivos.

---
### 3.2 Pares de confusión más frecuentes

Con 13 clases hay 156 posibles pares de error. Los más frecuentes revelan si los errores son inter-subtipo (confunde el subtipo TDAH principal) o intra-subtipo (acierta el subtipo pero falla en la comorbilidad).

In [ ]:
plot_confusion_pairs(y_te13, pred13, le13.classes_,
                     'RF — 13 clases', top_n=12)

📋 **Interpretación**

Si los pares más frecuentes comparten el mismo subtipo base (ej. *TDAH inatento* → *TDAH inatento + TND*), el modelo está acertando el diagnóstico principal pero fallando en identificar la comorbilidad. Esto es clínicamente menos grave que confundir subtipos diferentes.

La presencia de pares inter-subtipo (ej. *TDAH inatento* → *TDAH combinado*) en el modelo de 13 clases sería inesperada dado el rendimiento del modelo de 4 clases y sugeriría que esa distinción es más difícil en las variantes con comorbilidad.

---
### 3.3 Análisis de errores por subtipo principal

¿Los errores del modelo de 13 clases son principalmente intra-subtipo (comorbilidad incorrecta) o inter-subtipo (subtipo principal incorrecto)?

In [ ]:
def tipo4_from_label(e):
    e = str(e).lower()
    if 'típico'      in e: return 'Desarrollo típico'
    if 'inatento'    in e: return 'TDAH inatento'
    if 'hiperactivo' in e: return 'TDAH hiperactivo/impulsivo'
    return 'TDAH combinado'

true_labels13 = le13.inverse_transform(y_te13)
pred_labels13 = le13.inverse_transform(pred13)

true_tipo = np.array([tipo4_from_label(l) for l in true_labels13])
pred_tipo = np.array([tipo4_from_label(l) for l in pred_labels13])

correct_subtype    = (true_tipo == pred_tipo)
correct_full       = (y_te13 == pred13)
intra_subtype_err  = (~correct_full) & correct_subtype   # falla comorbilidad, acierta subtipo
inter_subtype_err  = (~correct_full) & (~correct_subtype) # falla subtipo también

n_total   = len(y_te13)
n_correct = correct_full.sum()
n_intra   = intra_subtype_err.sum()
n_inter   = inter_subtype_err.sum()

print(f"Total test:           {n_total}")
print(f"Correctos (13 cls):   {n_correct} ({n_correct/n_total*100:.1f}%)")
print(f"Errores totales:      {n_total - n_correct} ({(n_total-n_correct)/n_total*100:.1f}%)")
print(f"  ↳ Intra-subtipo:    {n_intra}  ({n_intra/(n_total-n_correct)*100:.1f}% de los errores)")
print(f"    (acierta subtipo, falla comorbilidad)")
print(f"  ↳ Inter-subtipo:    {n_inter}  ({n_inter/(n_total-n_correct)*100:.1f}% de los errores)")
print(f"    (falla subtipo principal)")

# Gráfica de proporción
fig, ax = plt.subplots(figsize=(9, 4))
cats   = ['Correctos\n(13 etiquetas)', 'Error intra-subtipo\n(falla comorbilidad)',
          'Error inter-subtipo\n(falla subtipo)']
vals   = [n_correct, n_intra, n_inter]
colors = [C_OK, C_ORG, C_ERR]
bars   = ax.bar(cats, vals, color=colors, edgecolor='white', width=0.55)
for bar, val in zip(bars, vals):
    pct = val / n_total * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Número de casos (test set)', fontsize=10)
ax.set_title('RF — 13 clases\nDescomposición de errores: intra-subtipo vs inter-subtipo', pad=8)
ax.set_ylim(0, max(vals) * 1.25)
plt.tight_layout()
plt.show()

📋 **Interpretación**

Esta descomposición es la métrica más informativa para evaluar la utilidad clínica del modelo de 13 clases:

- **Errores intra-subtipo** (alto porcentaje esperado): el modelo acierta el subtipo TDAH principal pero no detecta la comorbilidad. En la práctica clínica, esto es preferible al error inter-subtipo — el diagnóstico principal es correcto.

- **Errores inter-subtipo** (bajo porcentaje esperado): el modelo falla en el subtipo principal. Si este porcentaje es bajo (<10% de los errores totales), el modelo de 13 clases es consistente con el de 4 clases y sus errores son de "segundo orden" (comorbilidad).

Esta distinción tiene implicaciones para el uso del modelo: si se despliega como apoyo clínico, los errores intra-subtipo requerirían verificación adicional de la comorbilidad, mientras que los inter-subtipo necesitarían reevaluación completa del subtipo.

---
### 3.4 Confianza del modelo

In [ ]:
plot_confidence(y_te13, pred13, proba13, le13.classes_,
               'RF — 13 clases')

📋 **Interpretación**

Con 13 clases el azar teórico es 0.077 (1/13). La confianza máxima promedio del modelo para predicciones correctas refleja cuánto concentra la probabilidad en la clase ganadora. En modelos de alta precisión es esperable que los casos correctos tengan max_proba > 0.70 en promedio.

La comparación por clase es especialmente útil aquí: las clases con comorbilidades raras (pocos casos en el dataset) tienden a tener menor confianza media, tanto en aciertos como en errores — el modelo no tiene suficientes ejemplos para formar una representación robusta.

---
### 3.5 Perfil clínico de errores — clases principales con más errores

In [ ]:
# Filtrar clases con al menos 1 error para el perfil
error_mask13  = y_te13 != pred13
classes_w_err = [i for i in range(len(le13.classes_))
                 if ((y_te13 == i) & error_mask13).sum() > 0]
labels_w_err  = le13.classes_[classes_w_err]

y_te13_sub  = y_te13.copy()
pred13_sub  = pred13.copy()
# Remap para solo mostrar clases con errores (todos los casos de esas clases)
mask_sel     = np.isin(y_te13, classes_w_err)
y_plot       = y_te13[mask_sel]
pred_plot    = pred13[mask_sel]

# Re-encodear a índices 0..N
from sklearn.preprocessing import LabelEncoder as LE2
le_sub = LE2()
y_plot_enc    = le_sub.fit_transform(y_plot)
pred_plot_enc = le_sub.transform(pred_plot)
labels_plot   = le13.classes_[le_sub.classes_]

X_te13_df = pd.DataFrame(
    X_te13.values if hasattr(X_te13, 'values') else X_te13,
    columns=FEATURES
)
X_te13_df_sel = X_te13_df[mask_sel].reset_index(drop=True)

plot_error_profiles(y_plot_enc, pred_plot_enc, X_te13_df_sel, labels_plot,
                    'RF — 13 clases')

📋 **Interpretación**

El heatmap de perfiles para el modelo de 13 clases permite verificar si los errores intra-subtipo tienen un perfil clínico distinto al de los casos correctamente clasificados de la misma etiqueta.

Si los errores de *TDAH combinado + TND* tienen Agresividad (BASC-3) similar a *TDAH combinado* sin TND → el modelo no puede separar la comorbilidad porque las escalas disponibles no la capturan diferenciadamente. Esto es una limitación de las features, no del modelo.

---
## Bloque 4 — Síntesis comparativa y conclusiones

In [ ]:
# ── Resumen cuantitativo ─────────────────────────────────────────────────
err4  = (y_te4  != pred4).sum()
err13 = (y_te13 != pred13).sum()

print("=" * 55)
print(f"  Modelo 4 clases  — errores: {err4:3d} / {len(y_te4)}  ({err4/len(y_te4)*100:.1f}%)")
print(f"  F1-macro test:    {f1_4:.4f}")
print("=" * 55)
print(f"  Modelo 13 clases — errores: {err13:3d} / {len(y_te13)}  ({err13/len(y_te13)*100:.1f}%)")
print(f"  F1-macro test:    {f1_13:.4f}")
print("=" * 55)

# Confianza media global
mp4  = proba4.max(axis=1)
mp13 = proba13.max(axis=1)
print(f"\n  Confianza media (max_proba):")
print(f"    4 clases  — correctos: {mp4[y_te4==pred4].mean():.3f}  |  errores: {mp4[y_te4!=pred4].mean():.3f}")
print(f"    13 clases — correctos: {mp13[y_te13==pred13].mean():.3f}  |  errores: {mp13[y_te13!=pred13].mean():.3f}")


---
### Conclusiones del análisis de errores

**Modelo de 4 clases (RF, F1-macro ≈ 0.97):**
Los errores son escasos y se concentran en los pares de subtipos más cercanos clínicamente — principalmente entre *TDAH inatento* y *TDAH combinado*, que comparten el síntoma cardinal de inatención. Los casos mal clasificados tienden a tener perfiles mixtos (puntuaciones BASC-3 elevadas en múltiples dimensiones), lo que confirma que los errores son genuinamente ambiguos, no artefactos del modelo.

**Modelo de 13 clases (RF, F1-macro ≈ 0.95):**
La mayoría de los errores son *intra-subtipo* — el modelo acierta el subtipo TDAH principal pero falla en la comorbilidad. Esto tiene dos implicaciones: (a) las escalas BASC-3 disponibles no siempre discriminan suficientemente entre variantes del mismo subtipo con y sin comorbilidad, y (b) el modelo de 13 clases es clínicamente coherente con el de 4 clases (no introduce errores de subtipo nuevos).

**Implicaciones metodológicas:**
- Los errores de alta confianza (max_proba > 0.80) merecen revisión clínica independiente — son los casos donde el modelo puede inducir a error sin señal de advertencia.
- Para un despliegue clínico real, se recomendaría reportar la distribución de probabilidades completa (no solo la clase ganadora) para que el clínico pueda identificar cuando el modelo está cerca de un umbral de decisión.
- La adición de variables que capturen directamente la dimensión conductual oposicionista (para TND) o social/comunicativa (para TEA) podría reducir significativamente los errores intra-subtipo en el modelo de 13 clases.

---
*Análisis de errores — Proyecto EDA-TDAH · Mayo 2026*